# Tend — Privacy-First Life-Admin Concierge

Tend is a privacy-first life-admin concierge built as a Capstone project for the Kaggle Concierge track. It processes incoming personal emails, schedules action items on a calendar, drafts response drafts, and compiles a daily briefing.

## 1. Problem Statement & Architecture
- **Problem**: AI agents require access to highly sensitive personal data (emails, calendar) to manage life-admin. This exposes users to major risks of data exfiltration, unauthorized actions (e.g. sending emails autonomously), and indirect prompt injection attacks (where emails contain hidden instructions that hijack the agent).
- **Architecture**: Tend uses a strict multi-agent pipeline orchestrated sequentially. Personal data is never read directly from the disk; it is accessed only through a custom MCP server which filters out internal metadata. A dedicated security plugin screens LLM payloads and tool responses, while the action layer enforces human confirmation and recipient allowlists.

## 2. Core Concepts Mapping
This project explicitly implements four core agent development concepts:
1. **Multi-Agent ADK**: Sequential orchestration running parallel triage classifiers (`ParallelAgent`) and a self-reviewing briefing refinement loop (`LoopAgent`).
   - *Code pointer*: [`agents/orchestrator.py`](file:///c:/Users/alokk/agy2-projects/google-cloud-serverless-app/tend/agents/orchestrator.py)
2. **Custom MCP Data Server**: A read-only data boundary serving messages and calendar events while redacting evaluation flags.
   - *Code pointer*: [`mcp_server/server.py`](file:///c:/Users/alokk/agy2-projects/google-cloud-serverless-app/tend/mcp_server/server.py)
3. **Custom Agent Skill**: Enforces action-item extraction structures and reply tone templates.
   - *Code pointer*: [`skills/house_style/SKILL.md`](file:///c:/Users/alokk/agy2-projects/google-cloud-serverless-app/tend/skills/house_style/SKILL.md)
4. **Security Guardrails**: Implements PII redaction, indirect prompt injection defense, a recipient allowlist, and a human confirmation gate.
   - *Code pointer*: [`security/guardrails.py`](file:///c:/Users/alokk/agy2-projects/google-cloud-serverless-app/tend/security/guardrails.py) and [`mcp_server/server.py`](file:///c:/Users/alokk/agy2-projects/google-cloud-serverless-app/tend/mcp_server/server.py)

### Step 1: Install Dependencies
Install Google ADK, the MCP library, and python-dotenv in the notebook kernel.

In [ ]:
!pip install -q google-adk mcp python-dotenv

### Step 2: Load Kaggle Secrets
Retrieve the Gemini API key from Kaggle Secrets if running on the Kaggle platform. If not available, the pipeline automatically falls back to the mocked-model execution path.

In [ ]:
import os
try:
    from kaggle_secrets import UserSecretsClient
    user_secrets = UserSecretsClient()
    api_key = user_secrets.get_secret("GEMINI_API_KEY")
    if api_key:
        os.environ["GEMINI_API_KEY"] = api_key
        print("Successfully loaded GEMINI_API_KEY from Kaggle Secrets.")
except Exception as e:
    print("Kaggle Secrets not found or unavailable. Running in mocked-model fallback mode.")

### Step 3: Run the Concierge Pipeline
Execute the main entry point to run the triage, scheduling, drafting, and briefing pipeline on the synthetic data.

In [ ]:
!python main.py

### Step 4: Run the Security Demo
Execute the security demo script to run tests against the adversarial email injection payload and verify all four security guardrails trigger.

In [ ]:
!python demo_security.py